# Manual Clining

In [1]:
# import pandas as pd
# import json
# import re

# # Load the data
# file_path_processed = '/Users/hasanuddinm/Downloads/dicoding/tfx-practice/data/Political_Bias.csv'
# data = pd.read_csv(file_path_processed)

# # Load the mapping from JSON
# with open('/Users/hasanuddinm/Downloads/dicoding/tfx-practice/mapping.json', 'r') as f:
#     mapping = json.load(f)

# # Convert all column names to lowercase
# data.columns = [col.lower() for col in data.columns]

# # Function to clean text
# def clean_text(text):
#     if isinstance(text, str):
#         # Convert to lowercase
#         text = text.lower()
#         # Remove unusual characters
#         text = re.sub(r'[â€”â€™â—]', '', text)
#     return text

# # Apply the mapping to the bias column
# data['bias'] = data['bias'].map(mapping)

# # Clean the text column
# data['text'] = data['text'].apply(clean_text)

# # Remove rows with any NA values
# data = data.dropna()

# # Select only the 'text' and 'bias' columns
# data = data[['text', 'bias']]

# # Save the processed data to a new CSV file
# output_file_path = '/Users/hasanuddinm/Downloads/dicoding/tfx-practice/data/Political_Bias.csv'
# data.to_csv(output_file_path, index=False)

# print(f"Processed file saved to {output_file_path}")

In [2]:
# import os
# import csv

# def clean_csv_files(input_dir, output_dir):
#     if not os.path.exists(output_dir):
#         os.makedirs(output_dir)
    
#     for filename in os.listdir(input_dir):
#         if filename.endswith('.csv'):
#             input_path = os.path.join(input_dir, filename)
#             output_path = os.path.join(output_dir, filename)
            
#             with open(input_path, 'rb') as infile, open(output_path, 'w', encoding='utf-8', newline='') as outfile:
#                 reader = csv.reader(infile)
#                 writer = csv.writer(outfile)
                
#                 for row in reader:
#                     cleaned_row = [cell.decode('utf-8', errors='ignore') for cell in row]
#                     writer.writerow(cleaned_row)


In [3]:
# clean_csv_files("data", "cleaned_data")


# IMPORT LIBRARY

In [4]:
import tensorflow as tf
from tfx.components import CsvExampleGen, StatisticsGen, SchemaGen, ExampleValidator, Transform, Trainer, Tuner
from tfx.proto import example_gen_pb2
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext
import os

2025-03-28 04:41:42.637084: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


# Set Variable

In [5]:
PIPELINE_NAME = "political-bias-pipeline"
SCHEMA_PIPELINE_NAME = "political-bias-tfdv-schema"

#Directory untuk menyimpan artifact yang akan dihasilkan
PIPELINE_ROOT = os.path.join('yusrilhasan-pipelines', PIPELINE_NAME)

# Path to a SQLite DB file to use as an MLMD storage.
METADATA_PATH = os.path.join('metadata', PIPELINE_NAME, 'metadata.db')

# Output directory where created models from the pipeline will be exported.
SERVING_MODEL_DIR = os.path.join('yusrilhasan-serving_model_dir', PIPELINE_NAME)

# from absl import logging
# logging.set_verbosity(logging.INFO)

In [6]:
DATA_ROOT = "data"

In [7]:
interactive_context = InteractiveContext(pipeline_root=PIPELINE_ROOT)

In [8]:
output = example_gen_pb2.Output(
    split_config = example_gen_pb2.SplitConfig(splits=[
        example_gen_pb2.SplitConfig.Split(name="train", hash_buckets=8),
        example_gen_pb2.SplitConfig.Split(name="eval", hash_buckets=2)
    ])
)
c_input = example_gen_pb2.Input(splits=[
                      example_gen_pb2.Input.Split(name='data', pattern='*.csv')
                     ])
example_gen = CsvExampleGen(input_base=DATA_ROOT, output_config=output, input_config=c_input)

In [9]:
interactive_context.run(example_gen)

ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 274
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [10]:
statistics_gen = StatisticsGen(
    examples=example_gen.outputs["examples"]
)
 
 
interactive_context.run(statistics_gen)

ExecutionResult(
    component_id: StatisticsGen
    execution_id: 275
    outputs:
        statistics: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=StatisticsGen, output_key=statistics, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [11]:
interactive_context.show(statistics_gen.outputs["statistics"])

In [12]:
schema_gen = SchemaGen(    statistics=statistics_gen.outputs["statistics"]
)
interactive_context.run(schema_gen)


ExecutionResult(
    component_id: SchemaGen
    execution_id: 276
    outputs:
        schema: OutputChannel(artifact_type=Schema, producer_component_id=SchemaGen, output_key=schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [13]:
interactive_context.show(schema_gen.outputs["schema"])


,Type,Presence,Valency,Domain
Feature name,,,,
'bias',INT,required,,-
'text',BYTES,required,,-


In [14]:
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs['statistics'],
    schema=schema_gen.outputs['schema']
)
interactive_context.run(example_validator)


ExecutionResult(
    component_id: ExampleValidator
    execution_id: 277
    outputs:
        anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=ExampleValidator, output_key=anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [15]:
interactive_context.show(example_validator.outputs['anomalies'])


# Set Transformer

In [16]:
TRANSFORM_MODULE_FILE = "political_bias_transform.py"


In [17]:
%%writefile {TRANSFORM_MODULE_FILE}
import tensorflow as tf
LABEL_KEY = "bias"
FEATURE_KEY = "text"
def transformed_name(key):
    """Renaming transformed features"""
    return key + "_xf"
def preprocessing_fn(inputs):
    """
    Preprocess input features into transformed features
    
    Args:
        inputs: map from feature keys to raw features.
    
    Return:
        outputs: map from feature keys to transformed features.    
    """
    
    outputs = {}
    
    outputs[transformed_name(FEATURE_KEY)] = tf.strings.lower(inputs[FEATURE_KEY])
    
    # Ensure the label tensor has the correct shape
    labels = tf.one_hot(inputs[LABEL_KEY], depth=5)
    outputs[transformed_name(LABEL_KEY)] = tf.squeeze(labels, axis=-2)
    # outputs[transformed_name(LABEL_KEY)] = tf.one_hot(inputs[LABEL_KEY], depth=5)
    
    return outputs

Overwriting political_bias_transform.py


In [18]:
transform  = Transform(
    examples=example_gen.outputs['examples'],
    schema= schema_gen.outputs['schema'],
    module_file=os.path.abspath(TRANSFORM_MODULE_FILE)
)
interactive_context.run(transform)

running bdist_wheel
running build
running build_py
creating build
creating build/lib
copying political_bias_tuner.py -> build/lib
copying political_bias_transform.py -> build/lib
copying prediction.py -> build/lib
copying app.py -> build/lib
copying political_bias_trainer.py -> build/lib
copying main.py -> build/lib


/usr/local/lib/python3.9/site-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


installing to /var/folders/jr/pxqtskjn2l7bq4p9v4wh9wrw72vpzb/T/tmpe9pn4500
running install
running install_lib
copying build/lib/political_bias_tuner.py -> /var/folders/jr/pxqtskjn2l7bq4p9v4wh9wrw72vpzb/T/tmpe9pn4500
copying build/lib/political_bias_transform.py -> /var/folders/jr/pxqtskjn2l7bq4p9v4wh9wrw72vpzb/T/tmpe9pn4500
copying build/lib/prediction.py -> /var/folders/jr/pxqtskjn2l7bq4p9v4wh9wrw72vpzb/T/tmpe9pn4500
copying build/lib/app.py -> /var/folders/jr/pxqtskjn2l7bq4p9v4wh9wrw72vpzb/T/tmpe9pn4500
copying build/lib/political_bias_trainer.py -> /var/folders/jr/pxqtskjn2l7bq4p9v4wh9wrw72vpzb/T/tmpe9pn4500
copying build/lib/main.py -> /var/folders/jr/pxqtskjn2l7bq4p9v4wh9wrw72vpzb/T/tmpe9pn4500
running install_egg_info
running egg_info
creating tfx_user_code_Transform.egg-info
writing tfx_user_code_Transform.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Transform.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Transform.egg-info/top_level.txt


INFO:tensorflow:Assets written to: yusrilhasan-pipelines/political-bias-pipeline/Transform/transform_graph/278/.temp_path/tftransform_tmp/a8b2818939a0456c98eb35c7754de697/assets


INFO:tensorflow:tensorflow_decision_forests is not available.


...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


ExecutionResult(
    component_id: Transform
    execution_id: 278
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [19]:
def gzip_reader_fn(filenames):
    """Loads compressed data"""
    return tf.data.TFRecordDataset(filenames, compression_type='GZIP')

In [20]:
TRAINER_MODULE_FILE = "political_bias_trainer.py"

In [21]:
%%writefile {TRAINER_MODULE_FILE}
import tensorflow as tf
import tensorflow_transform as tft 
from tensorflow.keras import layers
import os  
import tensorflow_hub as hub
from tfx.components.trainer.fn_args_utils import FnArgs
 
LABEL_KEY = "bias"
FEATURE_KEY = "text"
NUM_CLASSES = 5  # Update this to match your number of categories

 
def transformed_name(key):
    """Renaming transformed features"""
    return key + "_xf"
 
def gzip_reader_fn(filenames):
    """Loads compressed data"""
    return tf.data.TFRecordDataset(filenames, compression_type='GZIP')
 
 
def input_fn(file_pattern, 
             tf_transform_output,
             num_epochs,
             batch_size=64)->tf.data.Dataset:
    """Get post_tranform feature & create batches of data"""
    
    # Get post_transform feature spec
    transform_feature_spec = (
        tf_transform_output.transformed_feature_spec().copy())
    
    # create batches of data
    dataset = tf.data.experimental.make_batched_features_dataset(
        file_pattern=file_pattern,
        batch_size=batch_size,
        features=transform_feature_spec,
        reader=gzip_reader_fn,
        num_epochs=num_epochs,
        label_key = transformed_name(LABEL_KEY))
    return dataset
 
# os.environ['TFHUB_CACHE_DIR'] = '/hub_chace'
# embed = hub.KerasLayer("https://tfhub.dev/google/universal-sentence-encoder/4")
 
# Vocabulary size and number of words in a sequence.
VOCAB_SIZE = 5000
SEQUENCE_LENGTH = 100
 
 
embedding_dim=16
def model_builder():
    """Build machine learning model"""
    # Get preprocessed input directly
    inputs = tf.keras.Input(shape=(1,), name=transformed_name(FEATURE_KEY), dtype=tf.string)
    
    # Use embedding with string input
    word_vectors = tf.keras.layers.Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=embedding_dim,
        input_length=SEQUENCE_LENGTH
    )(tf.strings.to_hash_bucket_fast(inputs, VOCAB_SIZE))
    
    x = tf.keras.layers.GlobalAveragePooling1D()(word_vectors)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    
    
    model = tf.keras.Model(inputs=inputs, outputs = outputs)
    
    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(0.01),
        metrics=tf.keras.metrics.CategoricalAccuracy(name='accuracy')
    
    )
    
    # print(model)
    model.summary()
    return model 
 
 
def _get_serve_tf_examples_fn(model, tf_transform_output):
    
    model.tft_layer = tf_transform_output.transform_features_layer()
    
    @tf.function
    def serve_tf_examples_fn(serialized_tf_examples):
        
        feature_spec = tf_transform_output.raw_feature_spec()
        
        feature_spec.pop(LABEL_KEY)
        
        parsed_features = tf.io.parse_example(serialized_tf_examples, feature_spec)
        
        transformed_features = model.tft_layer(parsed_features)
        
        # get predictions using the transformed features
        return model(transformed_features)
        
    return serve_tf_examples_fn
    
def run_fn(fn_args: FnArgs) -> None:
    
    log_dir = os.path.join(os.path.dirname(fn_args.serving_model_dir), 'logs')
    
    tensorboard_callback = tf.keras.callbacks.TensorBoard(
        log_dir = log_dir, update_freq='batch'
    )
    
    es = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', verbose=1, patience=10)
    mc = tf.keras.callbacks.ModelCheckpoint(fn_args.serving_model_dir, monitor='val_accuracy', mode='max', verbose=1, save_best_only=True)
    
    
    # Load the transform output
    tf_transform_output = tft.TFTransformOutput(fn_args.transform_graph_path)
    
    # Create batches of data
    train_set = input_fn(fn_args.train_files, tf_transform_output, 10)
    val_set = input_fn(fn_args.eval_files, tf_transform_output, 10)
    
    # Build the model
    model = model_builder()
    
    
    # Train the model
    model.fit(x = train_set,
            validation_data = val_set,
            callbacks = [tensorboard_callback, es, mc],
            steps_per_epoch = 10, 
            validation_steps= 10,
            epochs=10)
    signatures = {
        'serving_default':
        _get_serve_tf_examples_fn(model, tf_transform_output).get_concrete_function(
                                    tf.TensorSpec(
                                    shape=[None],
                                    dtype=tf.string,
                                    name='examples'))
    }
    model.save(fn_args.serving_model_dir, save_format='tf', signatures=signatures)


Overwriting political_bias_trainer.py


In [22]:
from tfx.proto import trainer_pb2
 
trainer  = Trainer(
    module_file=os.path.abspath("political_bias_trainer.py"),
    examples = transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=trainer_pb2.TrainArgs(splits=['train']),
    eval_args=trainer_pb2.EvalArgs(splits=['eval'])
)
interactive_context.run(trainer)


running bdist_wheel
running build
running build_py
creating build
creating build/lib
copying political_bias_tuner.py -> build/lib
copying political_bias_transform.py -> build/lib
copying prediction.py -> build/lib
copying app.py -> build/lib
copying political_bias_trainer.py -> build/lib
copying main.py -> build/lib


/usr/local/lib/python3.9/site-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


installing to /var/folders/jr/pxqtskjn2l7bq4p9v4wh9wrw72vpzb/T/tmp6hv2rr9o
running install
running install_lib
copying build/lib/political_bias_tuner.py -> /var/folders/jr/pxqtskjn2l7bq4p9v4wh9wrw72vpzb/T/tmp6hv2rr9o
copying build/lib/political_bias_transform.py -> /var/folders/jr/pxqtskjn2l7bq4p9v4wh9wrw72vpzb/T/tmp6hv2rr9o
copying build/lib/prediction.py -> /var/folders/jr/pxqtskjn2l7bq4p9v4wh9wrw72vpzb/T/tmp6hv2rr9o
copying build/lib/app.py -> /var/folders/jr/pxqtskjn2l7bq4p9v4wh9wrw72vpzb/T/tmp6hv2rr9o
copying build/lib/political_bias_trainer.py -> /var/folders/jr/pxqtskjn2l7bq4p9v4wh9wrw72vpzb/T/tmp6hv2rr9o
copying build/lib/main.py -> /var/folders/jr/pxqtskjn2l7bq4p9v4wh9wrw72vpzb/T/tmp6hv2rr9o
running install_egg_info
running egg_info
creating tfx_user_code_Trainer.egg-info
writing tfx_user_code_Trainer.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Trainer.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Trainer.egg-info/top_level.txt


writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
Copying tfx_user_code_Trainer.egg-info to /var/folders/jr/pxqtskjn2l7bq4p9v4wh9wrw72vpzb/T/tmp6hv2rr9o/tfx_user_code_Trainer-0.0+6f628e8bacc73aa15820df14e2769127c71f0bdabd16aa302bdec0d41cbccaa1-py3.9.egg-info
running install_scripts
creating /var/folders/jr/pxqtskjn2l7bq4p9v4wh9wrw72vpzb/T/tmp6hv2rr9o/tfx_user_code_Trainer-0.0+6f628e8bacc73aa15820df14e2769127c71f0bdabd16aa302bdec0d41cbccaa1.dist-info/WHEEL
creating '/var/folders/jr/pxqtskjn2l7bq4p9v4wh9wrw72vpzb/T/tmpx4i3m_i9/tfx_user_code_Trainer-0.0+6f628e8bacc73aa15820df14e2769127c71f0bdabd16aa302bdec0d41cbccaa1-py3-none-any.whl' and adding '/var/folders/jr/pxqtskjn2l7bq4p9v4wh9wrw72vpzb/T/tmp6hv2rr9o' to it
adding 'app.py'
adding 'main.py'
adding 'political_bias_trainer.py'
adding 'political_bias_transform.py'
adding 'political_bias_t

Instructions for updating:
Use `tf.data.Dataset.map(tf.io.parse_example(...))` instead.


...

INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.
2025-03-28 04:43:26.997953: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'examples' with dtype string and shape [?]
	 [[{{node examples}}]]
2025-03-28 04:43:27.072581: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'text' with dtype string and shape [?,1]
	 [[{{node text}}]]
2025-03-28 04:43:27.077871: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype string and shape

INFO:tensorflow:Assets written to: yusrilhasan-pipelines/political-bias-pipeline/Trainer/model/279/Format-Serving/assets


INFO:tensorflow:Assets written to: yusrilhasan-pipelines/political-bias-pipeline/Trainer/model/279/Format-Serving/assets


ExecutionResult(
    component_id: Trainer
    execution_id: 279
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Trainer, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        model_run: OutputChannel(artifact_type=ModelRun, producer_component_id=Trainer, output_key=model_run, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [23]:
from tfx.dsl.components.common.resolver import Resolver 
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy 
from tfx.types import Channel 
from tfx.types.standard_artifacts import Model, ModelBlessing 
 
model_resolver = Resolver(
    strategy_class= LatestBlessedModelStrategy,
    model = Channel(type=Model),
    model_blessing = Channel(type=ModelBlessing)
).with_id('Latest_blessed_model_resolver')
 
interactive_context.run(model_resolver)

ExecutionResult(
    component_id: Latest_blessed_model_resolver
    execution_id: 280
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Latest_blessed_model_resolver, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        model_blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Latest_blessed_model_resolver, output_key=model_blessing, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [24]:
import tensorflow_model_analysis as tfma 
 
eval_config = tfma.EvalConfig(
    model_specs=[tfma.ModelSpec(label_key='bias_xf')],
    slicing_specs=[tfma.SlicingSpec()],
    metrics_specs=[
        tfma.MetricsSpec(
            metrics=[
                tfma.MetricConfig(class_name='ExampleCount'),
                tfma.MetricConfig(class_name='Accuracy'),
                tfma.MetricConfig(class_name='AUC')
            ]
        )
    ]
 
)

In [25]:
from tfx.components import Evaluator
evaluator = Evaluator(
    examples=transform.outputs['transformed_examples'],
    model=trainer.outputs['model'],
    baseline_model=model_resolver.outputs['model'],
    eval_config=eval_config)
 
interactive_context.run(evaluator)

ExecutionResult(
    component_id: Evaluator
    execution_id: 281
    outputs:
        evaluation: OutputChannel(artifact_type=ModelEvaluation, producer_component_id=Evaluator, output_key=evaluation, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Evaluator, output_key=blessing, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [26]:
import tensorflow_model_analysis as tfma

# Path to the evaluation results
eval_result_path = 'yusrilhasan-pipelines/political-bias-pipeline/Evaluator/evaluation/199'

# Load the evaluation results
eval_result = tfma.load_eval_result(eval_result_path)

# Render the slicing metrics
tfma.view.render_slicing_metrics(eval_result)

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


SlicingMetricsViewer(config={'weightedExamplesColumn': 'example_count'}, data=[{'slice': 'Overall', 'metrics':…

In [27]:
from tfx.components import Pusher 
from tfx.proto import pusher_pb2 
 
pusher = Pusher(
model=trainer.outputs['model'],
model_blessing=evaluator.outputs['blessing'],
push_destination=pusher_pb2.PushDestination(
    filesystem=pusher_pb2.PushDestination.Filesystem(
        base_directory='yusrilhasan-serving_model_dir/political-bias-detection-model'))
 
)
 
interactive_context.run(pusher)

ExecutionResult(
    component_id: Pusher
    execution_id: 282
    outputs:
        pushed_model: OutputChannel(artifact_type=PushedModel, producer_component_id=Pusher, output_key=pushed_model, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [28]:
TUNER_MODULE_FILE = "political_bias_tuner.py"

In [29]:
%%writefile {TUNER_MODULE_FILE}

from tensorflow.keras import layers
import tensorflow as tf
import tensorflow_transform as tft
import os  
import tensorflow_hub as hub
from tfx.components.trainer.fn_args_utils import FnArgs
import keras_tuner as kt

try:
    # Try the most common import paths
    try:
        from tfx.components.tuner.component import TunerFnResult
    except ImportError:
        try:
            from tfx.extensions.google_cloud_ai_platform.tuner.component import TunerFnResult
        except ImportError:
            try:
                from tfx.v1.components.tuner.component import TunerFnResult
            except ImportError:
                # Define our own if none of the imports work
                class TunerFnResult:
                    def __init__(self, tuner, fit_kwargs):
                        self.tuner = tuner
                        self.fit_kwargs = fit_kwargs
except Exception as e:
    print(f"Error importing TunerFnResult: {e}")
    # Define a basic version if all else fails
    class TunerFnResult:
        def __init__(self, tuner, fit_kwargs):
            self.tuner = tuner
            self.fit_kwargs = fit_kwargs
            
LABEL_KEY = "bias"
FEATURE_KEY = "text"
NUM_CLASSES = 5
VOCAB_SIZE = 1000
embedding_dim = 16

def transformed_name(key):
    """Renaming transformed features"""
    return key + "_xf"

def gzip_reader_fn(filenames):
    """Loads compressed data"""
    return tf.data.TFRecordDataset(filenames, compression_type='GZIP')

def input_fn(file_pattern, 
             tf_transform_output,
             num_epochs=1,
             batch_size=64):
    """Get post_transform feature & create batches of data"""
    
    transform_feature_spec = (
        tf_transform_output.transformed_feature_spec().copy())
    
    dataset = tf.data.experimental.make_batched_features_dataset(
        file_pattern=file_pattern,
        batch_size=batch_size,
        features=transform_feature_spec,
        reader=gzip_reader_fn,
        num_epochs=num_epochs,
        label_key=transformed_name(LABEL_KEY))
    return dataset

def model_builder(hp):
    """Build machine learning model with minimal tuning"""
    inputs = tf.keras.Input(shape=(1,), name=transformed_name(FEATURE_KEY), dtype=tf.string)
    
    # Hash the text to integer indices
    hashed_text = tf.strings.to_hash_bucket_fast(inputs, VOCAB_SIZE)
    
    # Reshape to ensure consistent shape
    reshaped_text = tf.reshape(hashed_text, [-1, 1])
    
    # Embedding layer
    word_vectors = tf.keras.layers.Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=embedding_dim
    )(reshaped_text)
    
    x = tf.keras.layers.GlobalAveragePooling1D()(word_vectors)
    
    # Only tune one hyperparameter with explicit default value
    units = hp.Choice('units', values=[32, 64, 128], default=64)
    
    x = tf.keras.layers.Dense(units, activation='relu')(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    # Use a default learning rate to avoid None comparison
    learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4], default=1e-2)
    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        metrics=['accuracy']
    )
    return model

# Critical fix for the specific error
class CustomTuner(kt.RandomSearch):
    """Custom tuner that avoids the None comparison issue"""
    def get_best_models(self, num_models=1):
        """Override to avoid None comparison"""
        if num_models is None:
            num_models = 1
        return super().get_best_models(num_models)

def tuner_fn(fn_args: FnArgs):
    """Build the tuner using the KerasTuner API with fixes for the None comparison issue."""
    
    # Get transform output
    tf_transform_output = tft.TFTransformOutput(fn_args.transform_graph_path)
    
    # Create a custom tuner that handles None values
    tuner = CustomTuner(
        model_builder,
        objective='val_accuracy',
        max_trials=3,
        directory=fn_args.working_dir,
        project_name='political_bias_tuning'
    )
    
    # Create training dataset
    train_dataset = input_fn(
        file_pattern=fn_args.train_files,
        tf_transform_output=tf_transform_output,
        num_epochs=1,
        batch_size=64
    )
    
    # Create validation dataset
    eval_dataset = input_fn(
        file_pattern=fn_args.eval_files,
        tf_transform_output=tf_transform_output,
        num_epochs=1,
        batch_size=64
    )
    
    # Explicitly set non-None values for steps
    train_steps = 100 if fn_args.train_steps is None else fn_args.train_steps
    eval_steps = 50 if fn_args.eval_steps is None else fn_args.eval_steps
    
    return TunerFnResult(
        tuner=tuner,
        fit_kwargs={
            "x": train_dataset,
            "validation_data": eval_dataset,
            "steps_per_epoch": train_steps,  # Use explicit non-None value
            "validation_steps": eval_steps   # Use explicit non-None value
        }
    )


Overwriting political_bias_tuner.py


In [30]:
tuner = Tuner(
        module_file=os.path.join("political_bias_tuner.py"),
        examples=transform.outputs['transformed_examples'],
        transform_graph=transform.outputs['transform_graph'],
        schema=schema_gen.outputs['schema'],
        train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=500),
        eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=100)
    )

interactive_context.run(tuner, enable_cache=False)  # Disable cache to force fresh execution

Trial 3 Complete [00h 00m 02s]
val_accuracy: 0.5173010230064392

Best val_accuracy So Far: 0.5173010230064392
Total elapsed time: 00h 00m 06s
Results summary
Results in yusrilhasan-pipelines/political-bias-pipeline/.temp/283/political_bias_tuning
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 0 summary
Hyperparameters:
units: 64
learning_rate: 0.01
Score: 0.5173010230064392

Trial 2 summary
Hyperparameters:
units: 128
learning_rate: 0.001
Score: 0.5173010230064392

Trial 1 summary
Hyperparameters:
units: 128
learning_rate: 0.0001
Score: 0.508650541305542


ExecutionResult(
    component_id: Tuner
    execution_id: 283
    outputs:
        best_hyperparameters: OutputChannel(artifact_type=HyperParameters, producer_component_id=Tuner, output_key=best_hyperparameters, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        tuner_results: OutputChannel(artifact_type=TunerResults, producer_component_id=Tuner, output_key=tuner_results, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [31]:
# Add the evaluator to your pipeline components
pipeline_components = [
    example_gen,
    statistics_gen,
    schema_gen,
    example_validator,
    transform,
    trainer,
    evaluator,  # Ensure this is included
    pusher
]

In [32]:
!pipreqs --force

Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
INFO: Successfully saved requirements file in /Users/hasanuddinm/Downloads/dicoding/tfx-practice/requirements.txt


In [33]:
from tfx.orchestration import pipeline
from tfx.orchestration.local import local_dag_runner
from tfx.orchestration.metadata import sqlite_metadata_connection_config

pipeline = pipeline.Pipeline(
    pipeline_name='my_pipeline',
    pipeline_root='yusrilhasan-pipelines/political-bias-pipeline',
    components=pipeline_components,
    enable_cache=True,
    metadata_connection_config=sqlite_metadata_connection_config('storage/metadata.db')
)

local_dag_runner.LocalDagRunner().run(pipeline)

/usr/local/lib/python3.9/site-packages/tfx/orchestration/pipeline.py:408: UserWarning: Node Evaluator depends on the output of node Latest_blessed_model_resolver, but Latest_blessed_model_resolver is not included in the components of pipeline. Did you forget to add it?
  warnings.warn(


Processing ./yusrilhasan-pipelines/political-bias-pipeline/_wheels/tfx_user_code_Transform-0.0+173229cd995ecf985b5869826d3f06e1609829d8bc5b6335bd92c196d2bf11c7-py3-none-any.whl
Processing ./yusrilhasan-pipelines/political-bias-pipeline/_wheels/tfx_user_code_Transform-0.0+173229cd995ecf985b5869826d3f06e1609829d8bc5b6335bd92c196d2bf11c7-py3-none-any.whl
Processing ./yusrilhasan-pipelines/political-bias-pipeline/_wheels/tfx_user_code_Transform-0.0+173229cd995ecf985b5869826d3f06e1609829d8bc5b6335bd92c196d2bf11c7-py3-none-any.whl
INFO:tensorflow:Assets written to: yusrilhasan-pipelines/political-bias-pipeline/Transform/transform_graph/5/.temp_path/tftransform_tmp/a430e3aa719f4ba9bfcf73e6f21b7f98/assets


INFO:tensorflow:Assets written to: yusrilhasan-pipelines/political-bias-pipeline/Transform/transform_graph/5/.temp_path/tftransform_tmp/a430e3aa719f4ba9bfcf73e6f21b7f98/assets


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.
2025-03-28 04:44:48.694760: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'examples' with dtype string and shape [?]
	 [[{{node examples}}]]
2025-03-28 04:44:48.818470: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'text' with dtype string and shape [?,1]
	 [[{{node text}}]]
2025-03-28 04:44:48.830236: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype string and shape

INFO:tensorflow:Assets written to: yusrilhasan-pipelines/political-bias-pipeline/Trainer/model/6/Format-Serving/assets


INFO:tensorflow:Assets written to: yusrilhasan-pipelines/political-bias-pipeline/Trainer/model/6/Format-Serving/assets


In [35]:
import requests
from pprint import PrettyPrinter
 
pp = PrettyPrinter()
pp.pprint(requests.get("http://localhost:8500/v1/models/political-bias-detection-model").json())

{'model_version_status': [{'state': 'AVAILABLE',
                           'status': {'error_code': 'OK', 'error_message': ''},
                           'version': '1'}]}


In [36]:
from prediction import predict

# Define the server URL
SERVER_URL = 'http://localhost:8500/v1/models/political-bias-detection-model:predict'

# Input text
text = "This policy will benefit the economy while protecting our values. Said by the president of the United States. Donald Trump"

predict(SERVER_URL, text)


Prediction successful!
{
  "predictions": [
    [
      0.0,
      0.0368012041,
      0.00593836512,
      0.95726043,
      1.95311781e-10
    ]
  ]
}
Adjusted class index (1-indexed): 4
Predicted class: Right
Confidence: 95.73%

Probability distribution:
Far Left: 0.00%
Left: 3.68%
Center: 0.59%
Right: 95.73%
Far Right: 0.00%


'Right with 95.726043% confidence'